# Demo 6, T3: MCP from the agent

**Scope**: the agent stops carrying its own tool list. It connects to MCP servers, asks each one what tools it has (`tools/list`), and runs them (`tools/call`). We do it first against the T2 server, then change **only the URL** to reach a Databricks-provided MCP, `system.ai.sandbox`.

| Part | What happens |
|---|---|
| T3.1 | Connect to the T2 server with `DatabricksMCPClient` and list its tools |
| T3.2 | The raw JSON from T2 next to what the client gives back |
| T3.3 | The tutor's agent loop, with `search_wiki` coming from the server |
| T3.4 | Same client, new URL: `system.ai.sandbox` |
| T3.5 | Break it: a bad argument is a result, not an exception |

In T1 the tool list was fixed when the agent was built. Here the list is **whatever the server says when we ask**, so a tool added to a server (and the server redeployed) shows up the next time the agent builds its list, with no change to the agent [1].

**Before running**:
- T2 is done: the app `mcp-wiki-server` is running, and the service principal's credentials are in the secret scope `mcp-demo`.
- `common/governance_config.py` defines `GATEWAY_MODEL`, a model service whose model supports tool calling (as in T1).
- T3.4 needs `system.ai.sandbox`, which is under the Unity Gateway beta: an account admin must enable it, and you need access to it [2]. Without it, T3.4 says so and the rest of the notebook still works.

[n] refers to **References** at the end.

## T3.0 Setup

In [0]:
%pip install -U databricks-sdk databricks-mcp "mcp>=1.24,<2" "mlflow[databricks]>=3.1" openai nest-asyncio
dbutils.library.restartPython()

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import json
import os
import sys

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

import mlflow
import nest_asyncio
import openai
import pandas as pd
import requests
from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import NotFound
from databricks_mcp import DatabricksMCPClient

# DatabricksMCPClient.list_tools() and .call_tool() run `asyncio.run(...)` inside. A notebook already has an event loop running, so without this they fail with "asyncio.run() cannot be called from a running event loop".
nest_asyncio.apply()

from common.config import EXPERIMENT_PATH
from common.governance_config import GATEWAY_MODEL

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(EXPERIMENT_PATH)
mlflow.openai.autolog()  # every LLM call becomes a child span
w = WorkspaceClient()  # this notebook's own sign-in

# The LLM, as in T1: through a model service, not a serving endpoint.
os.environ["DATABRICKS_TOKEN"] = os.environ.get("DATABRICKS_TOKEN") or w.config.authenticate()["Authorization"].removeprefix("Bearer ")
gw = openai.OpenAI(api_key=os.environ["DATABRICKS_TOKEN"], base_url=f"{w.config.host.rstrip('/')}/ai-gateway/mlflow/v1")
TOOL_MODEL = GATEWAY_MODEL

# The T2 server, and the service principal that Apps accept (OAuth only) [3].
APP_NAME = "mcp-wiki-server"
SECRET_SCOPE = "mcp-demo"
try:
    MCP_URL = f"{w.apps.get(APP_NAME).url.rstrip('/')}/mcp"
except NotFound:
    raise RuntimeError(f"The app {APP_NAME} does not exist. Run T2 first.") from None

w_oauth = WorkspaceClient(
    host=w.config.host,
    client_id=dbutils.secrets.get(SECRET_SCOPE, "sp-client-id"),
    client_secret=dbutils.secrets.get(SECRET_SCOPE, "sp-client-secret"),
    auth_type="oauth-m2m",
)
w_oauth.config.oauth_token()  # fails here, early, if the credentials are wrong

SANDBOX_URL = f"{w.config.host.rstrip('/')}/ai-gateway/mcp-services/system.ai.sandbox"
print("Tool model:", TOOL_MODEL)
print("T2 server: ", MCP_URL)
print("Sandbox:   ", SANDBOX_URL)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


Tool model: main.new_python_wiki_agent.wiki_agent_llm
T2 server:  https://mcp-wiki-server-4475275120697622.aws.databricksapps.com/mcp
Sandbox:    https://dbc-c8b98f16-c6e2.cloud.databricks.com/ai-gateway/mcp-services/system.ai.sandbox


## T3.1 Connect and list the tools

`DatabricksMCPClient` takes a server URL and a `WorkspaceClient` that says who we are [1]. For the T2 server that has to be the **OAuth** client: Apps accept OAuth only (T2.3).

In [0]:
wiki_client = DatabricksMCPClient(server_url=MCP_URL, workspace_client=w_oauth)

wiki_tools = wiki_client.list_tools()
for t in wiki_tools:
    print("name:       ", t.name)
    print("description:", t.description)
    print("inputSchema:", json.dumps(t.inputSchema))

name:        search_wiki
description: Search the Python wiki by keywords. Returns the best-matching pages as {id, page_content, score}.
inputSchema: {"properties": {"query": {"description": "Keywords for the topic, e.g. 'add item list'", "title": "Query", "type": "string"}}, "required": ["query"], "title": "search_wikiArguments", "type": "object"}


**Look at**: the same three things as T2.4: a name, a description, an input schema. The client sent `tools/list` for us.

What if we hand it this notebook's own sign-in instead? T2.5 showed Apps refusing it. Here is what the client does with it:

In [0]:
try:
    DatabricksMCPClient(server_url=MCP_URL, workspace_client=w).list_tools()
    print("The call worked with the notebook's own sign-in.")
except Exception as e:
    print(type(e).__name__, "|", str(e)[:300])

ValueError | OAuth authentication is required for MCP servers hosted on Databricks Apps. Your current authentication method is not supported. Please use OAuth authentication instead. For more information: https://docs.databricks.com/aws/en/generative-ai/mcp/custom-mcp


**Look at**: the message. The client checks the sign-in when it is created: for an app URL with a non-OAuth sign-in it raises a `ValueError` before sending anything (this is in the `databricks-mcp` package, version 0.9.2 at the time of writing). That is the same rule the server enforces in T2.5, caught earlier and with a clearer message.

## T3.2 The raw messages, next to the client's

T2.4 sent `tools/list` and `tools/call` by hand. The client sends the same two messages. We prove it by sending them by hand again and comparing:

In [0]:
HEADERS = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}


def rpc(url, method, params=None, auth=None, id=1):
    body = {"jsonrpc": "2.0", "id": id, "method": method, **({"params": params} if params is not None else {})}
    return requests.post(url, json=body, headers={**HEADERS, **(auth or {})}, allow_redirects=False, timeout=60)


raw = rpc(MCP_URL, "tools/list", auth=w_oauth.config.authenticate()).json()["result"]["tools"][0]
via_client = wiki_tools[0]

print("raw tools/list   :", json.dumps(raw)[:300])
print("client list_tools:", json.dumps(via_client.model_dump(exclude_none=True))[:300])
assert (raw["name"], raw["description"], raw["inputSchema"]) == (via_client.name, via_client.description, via_client.inputSchema)
print("\nSame name, description and inputSchema.")

raw tools/list   : {"name": "search_wiki", "description": "Search the Python wiki by keywords. Returns the best-matching pages as {id, page_content, score}.", "inputSchema": {"properties": {"query": {"description": "Keywords for the topic, e.g. 'add item list'", "title": "Query", "type": "string"}}, "required": ["quer
client list_tools: {"name": "search_wiki", "description": "Search the Python wiki by keywords. Returns the best-matching pages as {id, page_content, score}.", "inputSchema": {"properties": {"query": {"description": "Keywords for the topic, e.g. 'add item list'", "title": "Query", "type": "string"}}, "required": ["quer

Same name, description and inputSchema.


In [0]:
ARGS = {"query": "add item list"}

raw_call = rpc(MCP_URL, "tools/call", {"name": "search_wiki", "arguments": ARGS}, auth=w_oauth.config.authenticate(), id=2).json()["result"]
res = wiki_client.call_tool("search_wiki", ARGS)

print("raw tools/call result keys   :", sorted(raw_call))
print("client CallToolResult fields :", sorted(k for k, v in res.model_dump(exclude_none=True).items()))
print("\nisError:", res.isError)
for block in res.content:
    print(block.type, "|", block.text[:160].replace("\n", " "))

raw tools/call result keys   : ['content', 'isError', 'structuredContent']
client CallToolResult fields : ['content', 'isError', 'structuredContent']

isError: False
text | {   "id": "loops",   "page_content": "`for` loops iterate over a sequence: `for n in [1, 2, 3]: print(n)`. `range(5)` gives 0 to 4. `while` repeats while a cond
text | {   "id": "lists",   "page_content": "Lists are ordered, changeable collections: `nums = [1, 2, 3]`. Add with `nums.append(4)`, remove with `nums.remove(2)`. In


**Look at**: the client returns the same `content`, `structuredContent` and `isError` as the raw reply. The client does no more than the JSON-RPC we sent by hand, plus the HTTP and sign-in. That is all MCP is at this layer, here or on localhost in Session 1.

## T3.3 The tutor's loop, with tools from MCP

The loop from T1, unchanged in shape: a tool is `{"spec", "run"}`, every call goes through one traced function. The only new code is the adapter: **MCP tool in, `{"spec", "run"}` out**. (`run_agent` is copied from T1; when T6 needs it too, it should move to `common/`.)

The adapter makes one decision worth reading. A failed MCP call is not an HTTP error: it is a normal reply with `isError: true` (T3.5). The adapter turns that into an exception, so the `TOOL` span is marked as failed and the model is told, instead of the failure looking like a success.

In [0]:
TUTOR_PROMPT = """You are a Python tutor for beginners. Use your tools when they help.
If the wiki does not cover a question, say so. Do not add links or sources that are not in the wiki."""
MAX_STEPS = 6


def result_text(res):
    """The text of an MCP result: the `content` blocks, else `structuredContent`."""
    text = "\n".join(getattr(b, "text", "") for b in res.content or [])
    return text or json.dumps(getattr(res, "structuredContent", None))


def mcp_runner(client, name):
    def run(**args):
        res = client.call_tool(name, args)
        if res.isError:  # a reply, not an HTTP error: raise so the span is marked as failed
            raise RuntimeError(result_text(res))
        return result_text(res)

    return run


def mcp_tools(client):
    """Ask the server what it has right now: {tool name: {"spec", "run"}}, the same shape as in T1."""
    tools = {}
    for t in client.list_tools():
        spec = {"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": t.inputSchema}}
        tools[t.name] = {"spec": spec, "run": mcp_runner(client, t.name)}
    return tools


def call_tool(tools, name, args):
    """Every tool call goes through here, so each one is a TOOL span."""
    with mlflow.start_span(name=name, span_type="TOOL") as span:
        span.set_inputs(args)
        result = tools[name]["run"](**args)
        span.set_outputs(result)
        return result


@mlflow.trace(span_type="AGENT")
def run_agent(question, tools, verbose=True):
    specs = [t["spec"] for t in tools.values()]
    messages = [{"role": "system", "content": TUTOR_PROMPT}, {"role": "user", "content": question}]
    calls = []
    for _ in range(MAX_STEPS):
        resp = gw.chat.completions.create(model=TOOL_MODEL, messages=messages, tools=specs, temperature=0.2)
        msg = resp.choices[0].message
        if not msg.tool_calls:
            return {"response": msg.content, "tool_calls": calls}
        messages.append(msg.model_dump(exclude_none=True))
        for tc in msg.tool_calls:
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            try:
                result = call_tool(tools, name, args)
            except Exception as e:  # the TOOL span is already marked as an error; tell the model too
                result = f"Error: {e}"
            calls.append({"tool": name, "args": args})
            if verbose:
                print(f"  -> {name}({args})\n  <- {str(result)[:150]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": result if isinstance(result, str) else json.dumps(result)})
    return {"response": "(stopped: too many tool calls)", "tool_calls": calls}


def try_questions(tools, questions):
    rows = []
    for q in questions:
        out = run_agent(q, tools, verbose=False)
        rows.append({
            "question": q,
            "tools used": ", ".join(c["tool"] for c in out["tool_calls"]) or "(none)",
            "answer (start)": out["response"][:140].replace("\n", " "),
        })
    display(pd.DataFrame(rows))


tools_wiki = mcp_tools(wiki_client)
print("Tools the agent got from the server:", list(tools_wiki))

Tools the agent got from the server: ['search_wiki']


In [0]:
QUESTIONS = [
    "How do I add an item to a list?",
    "What is a generic in Java?",
]
try_questions(tools_wiki, QUESTIONS)

question,tools used,answer (start)
How do I add an item to a list?,(none),"To add an item to a list in Python, you can use the `append()` method. Here's a simple example: ```python my_list = [1, 2, 3] my_list.appen"
What is a generic in Java?,search_wiki,"The search results did not return any relevant information about Java generics from the Python wiki. However, I can still provide a general"


[Trace(trace_id='tr-c6a9057765e07b0620195c156cd2df7f'), Trace(trace_id='tr-2777ea2e606fdb32c14e7ed6c54d08b4')]

**Look at**: the table, then the latest traces. The `TOOL` span for `search_wiki` looks the same as in T1, but the function now ran **in the app**, behind OAuth, not in this notebook. In the app's logs (**Compute > Apps > mcp-wiki-server**) you can see the request arrive [1].

The model's choices are its own, as in T1; the table shows what it did.

## T3.4 Change only the URL: `system.ai.sandbox`

A Databricks-provided MCP is reached the same way: same client, same `list_tools()`, same adapter. What changes is the URL (`.../ai-gateway/mcp-services/<catalog>.<schema>.<service>`) and **who may call**: here it is a Unity Catalog securable under Unity Gateway, so the caller needs access to it [2], and this notebook's own sign-in is enough (no service principal, no OAuth-only rule).

The docs say the sandbox "runs Python, SQL, or shell code in an isolated environment" and has no network egress, but do not list its tools. So we **ask**, which is the point of MCP:

In [0]:
SANDBOX_OK = False
try:
    sandbox_client = DatabricksMCPClient(server_url=SANDBOX_URL, workspace_client=w)
    sandbox_tools = sandbox_client.list_tools()
    SANDBOX_OK = True
    for t in sandbox_tools:
        print("name:       ", t.name)
        print("description:", (t.description or "")[:200])
        print("inputSchema:", json.dumps(t.inputSchema)[:300])
        print()
except Exception as e:
    print("Could not list the sandbox's tools:", type(e).__name__, "|", str(e)[:300])
    print("\nThe sandbox is under the Unity Gateway beta: an account admin must enable it, and you need access to it.")
    print("Everything else in this notebook works without it.")

name:        run_code
description: Execute code on Databricks compute. Supports Python, SQL, and sh.
inputSchema: {"type": "object", "properties": {"code": {"type": "string", "description": "The code to execute."}, "language": {"type": "string", "description": "Execution language: \"python\", \"sql\", or \"sh\" (default: \"python\")"}, "timeout_seconds": {"type": "integer", "description": "Maximum execution tim



**Look at**: tool names and schemas you did not write and that this notebook never hard-coded. Now call one by hand. The cell picks the tool with `python` in its name and fills its first required string parameter; if your sandbox's tools look different, set `TOOL_NAME` and `ARGS` yourself from what the cell above printed.

In [0]:
if SANDBOX_OK:
    py = next((t for t in sandbox_tools if "python" in t.name.lower()), sandbox_tools[0])
    required = py.inputSchema.get("required", [])
    string_params = [p for p in required if py.inputSchema["properties"].get(p, {}).get("type") == "string"]
    TOOL_NAME = py.name
    ARGS = {string_params[0]: "print(2 ** 10)"} if string_params else {}
    print("Calling", TOOL_NAME, "with", ARGS)
    res = sandbox_client.call_tool(TOOL_NAME, ARGS)
    print("isError:", res.isError)
    print(result_text(res)[:500])
else:
    print("Skipped: the sandbox is not available.")

Calling run_code with {'code': 'print(2 ** 10)'}
isError: True
Sandbox is still starting. Retry the sandbox call shortly; the same session will be reused.


Now the agent with **two clients, one tool list**: `search_wiki` from our server and the sandbox's tools from Unity Gateway. The loop does not know or care which server a tool came from.

In [0]:
if SANDBOX_OK:
    tools_all = {**tools_wiki, **mcp_tools(sandbox_client)}
    assert len(tools_all) == len(tools_wiki) + len(sandbox_tools), "two servers offer a tool with the same name"
    print("One tool list:", list(tools_all))
    try_questions(tools_all, [
        "How do I add an item to a list?",
        "What is 2 to the power of 100? Run it, don't guess.",
    ])
else:
    print("Skipped: the sandbox is not available.")

One tool list: ['search_wiki', 'run_code']


question,tools used,answer (start)
How do I add an item to a list?,(none),"To add an item to a list in Python, you can use the `append()` method. Here's an example: ```python my_list = [1, 2, 3] my_list.append(4) p"
"What is 2 to the power of 100? Run it, don't guess.","run_code, run_code, run_code, run_code, run_code",It seems like the sandbox is having issues starting. Let me try a different approach. You can calculate 2 to the power of 100 using Python.


[Trace(trace_id='tr-31c3810f2341e060611c64f3f6192705'), Trace(trace_id='tr-62aa57065469dec3876893e2519f9634')]

**Look at**: which tool each question used. The first needs the wiki, the second needs code to run. The model chose from the descriptions each server gave it. If two servers offered the same tool name, the second would silently replace the first in a dict, hence the assertion.

## T3.5 Break it: a bad argument is a result, not an exception

Call `search_wiki` without its required `query`:

In [0]:
res = wiki_client.call_tool("search_wiki", {})  # no exception is raised here
print("raised an exception? no")
print("isError:", res.isError)
print(result_text(res)[:300])

raised an exception? no
isError: True
Error executing tool search_wiki: 1 validation error for search_wikiArguments
query
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


**Look at**: the call **returned**. A client that only watches for exceptions, or for the HTTP status, would treat this as a success and pass the error text to the model as if it were wiki content. (T2 showed the raw reply: HTTP 200 with `isError: true`.)

That is why the adapter in T3.3 checks `isError` and raises. See the difference in a trace:

In [0]:
@mlflow.trace(span_type="AGENT")
def failing_call():
    try:
        return call_tool(tools_wiki, "search_wiki", {})
    except Exception as e:
        return f"Error: {e}"[:200]


print(failing_call())

Error: Error executing tool search_wiki: 1 validation error for search_wikiArguments
query
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://er


Trace(trace_id='tr-7a1625d90b55099c240974ad698b6a5b')

**Look at**: the latest trace. The `search_wiki` `TOOL` span is marked as an **error**, with the server's message, and the loop hands the model `Error: ...` so it can try again or tell the learner. Without the `isError` check the span would be green and the failure invisible.

## What T3 left open

- Nothing governs who may call these tools beyond "has `CAN USE`" on our app and "has access" to the sandbox. T5 grants and revokes both, adds Gateway controls on the sandbox, and looks at what the Gateway recorded.
- `get_exercise` is still a UC function called through a different client (`DatabricksFunctionClient`). T6 puts it in the same tool list as the MCP tools.
- T4 (notes only) covers MCP servers that live outside Databricks.

# References

Docs checked 2026-10-25.

1. [Use MCP tools in a Python agent](https://docs.databricks.com/aws/en/agents/custom-agents/use-mcp-tools): `DatabricksMCPClient(server_url, workspace_client)`, `list_tools()`, `call_tool()`, URL formats, checking `isError`, `mcp>=1.24,<2`
2. [Databricks-provided MCPs](https://docs.databricks.com/aws/en/agents/mcp-tools/built-in-mcp-services): `system.ai.sandbox`, Unity Gateway beta, UC securables
3. [Authentication and networking for MCP](https://docs.databricks.com/aws/en/agents/mcp-tools/authentication-networking): OAuth only for Apps servers; service principal with OAuth M2M for unattended callers